In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import csv
from sklearn.feature_selection import mutual_info_classif,VarianceThreshold
from sklearn.svm import LinearSVC
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import Lasso


from sklearn.feature_selection import f_classif, mutual_info_classif, chi2, VarianceThreshold, RFE
from sklearn.decomposition import PCA
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# import qnorm
from sklearn.preprocessing import QuantileTransformer, MinMaxScaler
print("Libraries loaded")

In [ ]:
cancer_type = "BRCA" # BRCA, COADREAD. GBMLGG, HNSC, KIPAN, LUAD, STES


FILTER FOR MATCHING SAMPLES IN METACLINICAL AND RNASEQ DATA

In [ ]:
metaclin = pd.read_csv(f'/path/Data/TCGA_{cancer_type}/gdac.broadinstitute.org_{cancer_type}.Merge_Clinical.Level_1.2016012800.0.0/{cancer_type}.merged_only_clinical_clin_format.txt', sep= '\t', header =0, dtype= 'string')
print("Metaclin.shape: ", metaclin.shape)
rnaseq = pd.read_csv(f'/path/Data/TCGA_{cancer_type}/gdac.broadinstitute.org_{cancer_type}.Merge_rnaseqv2__illuminahiseq_rnaseqv2__unc_edu__Level_3__RSEM_genes_normalized__data.Level_3.2016012800.0.0/{cancer_type}.rnaseqv2__illuminahiseq_rnaseqv2__unc_edu__Level_3__RSEM_genes_normalized__data.data.txt', sep= '\t', header =0, dtype={'Hybridization REF': 'string'})

rnaseq.drop(0, axis=0,inplace=True)
print("RNASeq.shape: ", rnaseq.shape) # (20531, 1213)

metaclin_barcodes = metaclin[metaclin['V1'] == 'patient.bcr_patient_barcode'].values.flatten()[1:]
print("Metaclin.barcodes: ", len(metaclin_barcodes)) 
rnaseq_barcodes = [col.lower() for col in rnaseq.columns]
rnaseq_barcodes = [col[:12] for col in rnaseq_barcodes]
rnaseq_barcodes = rnaseq_barcodes[1:]
print("RNASeq.barcodes: ", len(rnaseq_barcodes)) 
common = set(rnaseq_barcodes).intersection(metaclin_barcodes)
print("Common barcodes: ", len(common)) 
rnaseq.set_index('Hybridization REF', inplace=True) 
indexes = rnaseq.index
rnaseq = rnaseq[rnaseq.columns[rnaseq.columns.str[13:15] == "01"]] # filter for samples from tumor sites (should be majority of patients)
print("RNASeq.shape (tumor samples): ", rnaseq.shape) # (20531, 1093)
rnaseq = rnaseq[rnaseq.columns[rnaseq.columns.str[:12].str.lower().isin(common)]]
print("RNASeq.shape (common in metaclin): ", rnaseq.shape) # (20531, 1093)
rnaseq = rnaseq.transpose()
rnaseq.columns = indexes
print("RNASeq.shape: ", rnaseq.shape) # (1093, 20531)

cols = rnaseq.columns
for col in cols:
    rnaseq[col] = rnaseq[col].astype(float)
filtered_rnaseq_barcodes = [row.lower() for row in rnaseq.index]
filtered_rnaseq_barcodes = [row[:12] for row in filtered_rnaseq_barcodes]
print("Filtered barcodes: ", len(filtered_rnaseq_barcodes))
metaclin.set_index('V1', inplace=True) 
rowname = 'patient.bcr_patient_barcode'
filtered_metaclin = metaclin.loc[:, metaclin.loc[rowname].isin(filtered_rnaseq_barcodes)]
filtered_metaclin.columns = filtered_metaclin.loc[rowname]
filtered_metaclin = filtered_metaclin.drop(rowname, axis =0)
filtered_metaclin = filtered_metaclin.transpose()
filtered_metaclin = filtered_metaclin.replace(to_replace="living", value="alive")
filtered_metaclin = filtered_metaclin.loc[filtered_rnaseq_barcodes] # order according to rnaseq data
print("Metaclin.shape: ", filtered_metaclin.shape)
filtered_metaclin.to_csv(f'/path/Data/TCGA_{cancer_type}/filtered_metaclinical.txt', sep='\t', index=True)

REMOVE GENES WITH MOSTLY ZEROS

In [ ]:
# 1st step: remove genes not expressed in more than 99% of patients
# Remove lowly expressed genes
zero_gene_counts = np.sum(rnaseq <= 0, axis=0)  
print("zero_gene_counts: ", zero_gene_counts)
print("rnaseq shape: ", rnaseq.shape) 
nonzero_gene_counts = np.sum(rnaseq > 0, axis=0)  
genes_to_keep = nonzero_gene_counts.loc[lambda x : x > (rnaseq.shape[0]*0.99)]
print("genes to keep: ", genes_to_keep) 
frna_zerosrm = rnaseq.loc[:, genes_to_keep.index]
print("final rnaseq shape: ", frna_zerosrm.shape) 
frna_zerosrm.to_csv(f'/path/Data/TCGA_{cancer_type}/filtered_rnaseq_zeros_removed.txt', sep='\t', index=True)

USE LASSO REGULARIZATION TO KEEP GENES WITH HIGH VARIANCE ACROSS SAMPLES

In [ ]:
df = pd.read_csv(f"/path/Data/TCGA_{cancer_type}/filtered_rnaseq_zeros_removed.txt", sep= '\t', index_col =0, header =0)
df_labels = pd.read_csv(f"/path/Data/TCGA_{cancer_type}/filtered_metaclinical.txt", sep= '\t', index_col=0) 
metadata = "patient.vital_status"
df_labels = df_labels[metadata]
df_labels = df_labels.dropna()
print("Frequency of values: ", df_labels.value_counts())  
df.index = df.index.str[:12].str.lower()
df = df.loc[df_labels.index]
meta_label = df_labels.astype('category')
meta_label = meta_label.cat.codes
freq_counts = meta_label.value_counts()
weights = {i: 1/freq_counts[i] for i in range(len(freq_counts))}
print("Weights: ", weights)
np.random.seed(1)
print(freq_counts[0])
1/(freq_counts[0]/(freq_counts[0]+freq_counts[1]))
1/(freq_counts[1]/(freq_counts[0]+freq_counts[1]))

def select_features_with_L1(X, y):
    alpha = 1
    lasso = Lasso(alpha=alpha, fit_intercept=False, max_iter=100000)
    lasso.fit(X, y)
    return np.abs(lasso.coef_) 
scores_L1 = select_features_with_L1(df, meta_label)
np.count_nonzero(scores_L1)

np.random.seed(1)
lsvc = LinearSVC(C=1000, penalty="l1", max_iter=100000).fit(df, meta_label)
model = SelectFromModel(lsvc, prefit=True)
df_c1000 = model.transform(df)
print(df_c1000.shape)
df_c1000 = pd.DataFrame(df_c1000)
df_c1000.index = df.index
c1000_idx = model.get_support()
df_c1000.columns = df.columns[c1000_idx]

df_c1000.to_csv(f'/path/Data/TCGA_{cancer_type}/filtered_rnaseq_zerosremoved_lassoselectedc1000.txt', sep='\t', index=True)

LOG TRANSFORM TO NORMALIZE

In [ ]:
transformed_df = np.log2(df_c1000 + 1)
transformed_df.to_csv(f'/path/Data/TCGA_{cancer_type}/filtered_rnaseq_zerosremoved_lassoselectedc1000_logtransformed.txt', sep='\t', index=True)